In [1]:
%load_ext autoreload
%autoreload 2

In [6]:
import utils
from enumeration_v1 import Enumerator
import tqdm
from collections import defaultdict

In [44]:
B = 70
enum = Enumerator(B)
fb = enum.factor_base
fb

{5: (2, 1),
 13: (3, 2),
 17: (4, 1),
 29: (5, 2),
 37: (6, 1),
 41: (5, 4),
 53: (7, 2),
 61: (6, 5)}

In [45]:
factors = [5, 5, 17, 17, 13, 13]
res = enum.get_all_points_from_factorization_marked(factors)


def validate_points_from_factorizations_marked(res, factors, fb):
    for m, p in res:
        e = utils.IntegerComplex(1, 0)
        for mm, ff in zip(m, factors):
            e *= fb[ff].conjugate() if mm else fb[ff]
        assert e == p, f"{e=}, {p=}, {m=}, {factors=}, {fb=}"


validate_points_from_factorizations_marked(res, factors, fb)
enum.canonize_to_first_eighth(enum.get_all_points_from_factorization(factors))

[(1073, 264),
 (1020, 425),
 (884, 663),
 (817, 744),
 (943, 576),
 (975, 520),
 (1001, 468),
 (1105, 0),
 (855, 700),
 (952, 561),
 (1071, 272),
 (1104, 47),
 (1092, 169),
 (1100, 105)]

In [46]:
mult_points_base = enum.canonize_to_first_eighth(
    enum.get_all_points_from_factorization([p for p in fb for _ in range(2)])
)
print(len(mult_points_base))
# mult_points_base = [p for p in mult_points_base if p[1] != 0]
mult_points_base = [p for p in mult_points_base]
print(len(mult_points_base))

a_single_try_group = enum.get_all_points_from_factorization(
    [5, 13, 17, 61], conj_first_point=True
)


def proj(p):
    return (p.real * p.imag) ** 2


def canonize_to_first_eighth(points):
    points_set = set()
    rv = []
    for idx, p in points:
        x = abs(p.real)
        y = abs(p.imag)
        res_p = (max(x, y), min(x, y))
        if res_p in points_set:
            continue
        points_set.add(res_p)
        rv.append((idx, utils.IntegerComplex(res_p[0], res_p[1])))
    return rv


all_points = []
for i, p_mult in tqdm.tqdm(enumerate(mult_points_base)):
    for j, p in enumerate(a_single_try_group):
        p_ = p_mult * p
        all_points.append(((i, j), p_))

all_points_ = canonize_to_first_eighth(all_points)
all_projs = [(idx, proj(p)) for idx, p in all_points_]

ht = dict()
for i, proj_lhs in tqdm.tqdm(enumerate(all_projs), total=len(all_projs)):
    for proj_rhs in all_projs[:i]:
        key = proj_lhs[1] + proj_rhs[1]
        if key in ht:
            ht[key].append((proj_lhs[0], proj_rhs[0]))
        else:
            ht[key] = [(proj_lhs[0], proj_rhs[0])]

3281
3281


0it [00:00, ?it/s]

3281it [00:00, 95280.21it/s]
100%|██████████| 10368/10368 [01:30<00:00, 114.50it/s]


In [47]:
len(ht)
# all_points

53742527

In [ ]:
sols = {}
for k, v in tqdm.tqdm(ht.items(), total=len(ht)):
    if len(v) > 1:
        point_pairs = []

        for vv in v:
            vv0, vv1 = vv
            # print(vv)
            # print(
            #     mult_points_base[vv0[0]],
            #     a_single_try_group[vv0[1]],
            #     mult_points_base[vv1[0]],
            #     a_single_try_group[vv1[1]],
            # )
            p0 = mult_points_base[vv0[0]] * a_single_try_group[vv0[1]]
            p1 = mult_points_base[vv1[0]] * a_single_try_group[vv1[1]]
            point_pairs.append((p0, p1))
            # print(f"{p0=}, {p0[0]**2 + p0[1]**2}")
            # print(f"{p1=}, {p1[0]**2 + p1[1]**2}")
            # print(f"{proj(p0)+proj(p1)}")
        # utils.analyze_E4_sol(
        #     (point_pairs[0], point_pairs[1]), factorize_gaussian_integers=False
        # )
        sols[k] = point_pairs

        # assert False

100%|██████████| 53742527/53742527 [00:07<00:00, 7158472.04it/s]


In [52]:
len(sols)

1

In [69]:
from sympy import I, factorint, gcd, simplify, re, im
import sympy


def is_gaussian_integer(z):
    """Checks if a complex number has exactly integer real and imaginary parts."""
    # Using simplify() ensures complex fractions are reduced before checking
    z_simp = simplify(z)
    return re(z_simp).is_integer and im(z_simp).is_integer


def factor_gaussian_int(z):
    """Factors a Gaussian integer z into a list of Gaussian primes."""
    a, b = z.as_real_imag()
    norm = a**2 + b**2
    factors = []
    remaining = z

    # Factor the norm into standard integer primes
    for p, exponent in factorint(norm).items():
        for _ in range(exponent):
            # Case 1: p = 2 (Ramified)
            if p == 2:
                g_prime = 1 + I
            # Case 2: p ≡ 3 (mod 4) (Inert)
            elif p % 4 == 3:
                # p is its own Gaussian prime (norm p^2), so use it if it divides
                g_prime = p
            # Case 3: p ≡ 1 (mod 4) (Split)
            else:
                # p splits into (x+iy)(x-iy). Find one using GCD.
                # Find x such that x^2 + 1 ≡ 0 (mod p) and use gcd(p, x+I)
                from sympy.ntheory import sqrt_mod

                x = int(sqrt_mod(-1, p))  # Finds a root of x^2 ≡ -1 (mod p)
                g_prime = gcd(p, x + I)

            # Check if this prime or its conjugate divides our current number
            q = sympy.simplify(remaining / g_prime)
            print(q)
            if is_gaussian_integer(q):
                print("True")
                factors.append(g_prime)
                remaining = q
            else:
                print("False")

                # If the prime doesn't divide, its conjugate must
                g_prime_conj = g_prime.conjugate()
                factors.append(g_prime_conj)
                remaining = simplify(remaining / g_prime_conj)

    # The remaining part is a unit (1, -1, I, -I)
    if remaining != 1:
        factors.append(remaining)

    return factors


keys = list(sols.keys())


p = sols[keys[0]][0][0]
factor_gaussian_int(p[0] + I * p[1])

-785817263725 + 18230960518420*I
True
3331865198194 + 7449547660113*I
True
14113278056501/5 + 11567230122032*I/5
False
817249954274 + 597221120431*I
True
263551020757 - 96715970920*I
True
236954128754/13 - 984085004111*I/13
False
17494212387 + 7111468450*I
True
77088317998/17 + 10951661413*I/17
False
17494212387/17 + 7111468450*I/17
False
196178440 - 63757993*I
True
73566915/29 - 1108408186*I/29
False
4771908 + 4109143*I
True
32740591/37 + 19882950*I/37
False
161650 - 3233*I
True
630435/41 - 821182*I/41
False
3172 - 1403*I
True
-3477/53 - 25010*I/53
False
61
True
5 - 6*I
True
(5 - 6*I)**2/61
False


[2 + I,
 2 + I,
 2 - I,
 2 + 3*I,
 2 + 3*I,
 2 - 3*I,
 4 + I,
 4 - I,
 4 - I,
 2 + 5*I,
 2 - 5*I,
 6 + I,
 6 - I,
 4 + 5*I,
 4 - 5*I,
 2 + 7*I,
 2 - 7*I,
 5 + 6*I,
 5 + 6*I,
 5 - 6*I]

In [59]:
sp.factor?

Signature: sp.factor(f, *gens, deep=False, **args)
Docstring:
Compute the factorization of expression, ``f``, into irreducibles. (To
factor an integer into primes, use ``factorint``.)

There two modes implemented: symbolic and formal. If ``f`` is not an
instance of :class:`Poly` and generators are not specified, then the
former mode is used. Otherwise, the formal mode is used.

In symbolic mode, :func:`factor` will traverse the expression tree and
factor its components without any prior expansion, unless an instance
of :class:`~.Add` is encountered (in this case formal factorization is
used). This way :func:`factor` can handle large or symbolic exponents.

By default, the factorization is computed over the rationals. To factor
over other domain, e.g. an algebraic or finite field, use appropriate
options: ``extension``, ``modulus`` or ``domain``.

Examples

>>> from sympy import factor, sqrt, exp
>>> from sympy.abc import x, y

>>> factor(2*x**5 + 2*x**4*y + 4*x**3 + 4*x**2*y + 2*x + 2*